# 8-bit ALU — PYNQ-Z2 Hardware Validation

This notebook tests the VHDL ALU through two AXI GPIO IPs on the **PYNQ-Z2**.

**Important:** The bitstream is programmed separately through Vivado Hardware Manager (JTAG). This notebook uses a **remote Jupyter kernel running on the board**. It does not download an overlay.

| GPIO | Base | Channel 1 (`0x00`) | Channel 2 (`0x08`) |
|---|---|---|---|
| Command | `0x41200000` | 32-bit command | start bit 0, ack bit 1 |
| Status | `0x41210000` | 32-bit response | busy bit 0, done bit 1 |

Command: A bits 0–7; B bits 8–15; opcode bits 16–18. Response: result bits 0–7; carry bit 8; borrow bit 9; zero bit 10; overflow bit 11.

**Safety:** An AXI read to an unavailable peripheral may hang the kernel. Python polling deadlines cannot interrupt a blocked MMIO read. Confirm the correct bitstream, addresses, and PS–PL configuration before running.

## 1. Configure the notebook

Verify FCLK0 is at 50 MHz. Do not alter the clock automatically, because other peripherals may depend on it.

In [18]:
import time
from random import Random
from pynq import Clocks, MMIO

CMD_BASE = 0x41200000
STATUS_BASE = 0x41210000
MMIO_SIZE = 0x1000
DATA = 0x00
CHANNEL2 = 0x08
START, ACK = 0b01, 0b10
BUSY, DONE = 0b01, 0b10
OPCODES = {"ADD": 0b000, "AND": 0b001, "OR": 0b010, "SUB": 0b011}

frequency = float(Clocks.fclk0_mhz)
print(f"FCLK0: {frequency:.1f} MHz")
assert abs(frequency - 50.0) < 0.1, "Expected FCLK0 at 50 MHz"

FCLK0: 50.0 MHz


## 2. Access the hardware

The two command channels are configured as output-only in Vivado; the status channels are input-only. There is no need to write direction (`TRI`) registers in this notebook.

In [19]:
gpio_cmd = MMIO(CMD_BASE, MMIO_SIZE)
gpio_status = MMIO(STATUS_BASE, MMIO_SIZE)

initial_status = gpio_status.read(CHANNEL2) & 0b11
print(f"Initial status: busy={bool(initial_status & BUSY)}, done={bool(initial_status & DONE)}")
if initial_status != 0:
    print("Controller is not idle. Investigate before issuing a new command.")

Initial status: busy=False, done=False


## 3. ALU driver

The handshake is `idle → start pulse → done → read response → ack pulse → idle`. The controller holds `done` until acknowledged.

In [20]:
def read_status():
    """Read the busy and done bits."""
    return gpio_status.read(CHANNEL2) & (BUSY | DONE)


def wait_status(predicate, *, timeout_s=1.0, label="status"):
    """Wait for a status condition; does not prevent a stalled MMIO access."""
    deadline = time.monotonic() + timeout_s
    while True:
        status = read_status()
        if predicate(status):
            return status
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Timeout waiting for {label}: status=0b{status:02b}")


def execute_alu(a, b, operation, *, timeout_s=1.0):
    """Execute one transaction and return the result with all flags."""
    if type(a) is not int or type(b) is not int:
        raise TypeError("Operands must be integers")
    if not (0 <= a <= 255 and 0 <= b <= 255):
        raise ValueError("Operands must be in the range 0..255")
    if not isinstance(operation, str):
        raise TypeError("Operation must be a string")
    op = operation.upper()
    if op not in OPCODES:
        raise ValueError(f"Unknown operation: {operation!r}")

    command = a | (b << 8) | (OPCODES[op] << 16)

    # Do not start while a previous result is pending.
    gpio_cmd.write(CHANNEL2, 0)
    wait_status(lambda s: s == 0, timeout_s=timeout_s, label="idle before start")
    gpio_cmd.write(DATA, command)

    # Generate a start pulse using two AXI writes.
    gpio_cmd.write(CHANNEL2, START)
    gpio_cmd.write(CHANNEL2, 0)
    wait_status(lambda s: bool(s & DONE), timeout_s=timeout_s, label="done")

    response = gpio_status.read(DATA)

    # Acknowledge completion, then verify that done has cleared.
    gpio_cmd.write(CHANNEL2, ACK)
    gpio_cmd.write(CHANNEL2, 0)
    wait_status(lambda s: s == 0, timeout_s=timeout_s, label="idle after ack")

    return {
        "result": response & 0xFF,
        "carry": (response >> 8) & 1,
        "borrow": (response >> 9) & 1,
        "zero": (response >> 10) & 1,
        "overflow": (response >> 11) & 1,
    }

## 4. Independent Python reference model

The model implements unsigned carry and borrow, signed overflow, and the zero flag. Every test compares **all five fields**, not just the result.

In [21]:
def reference_alu(a, b, operation):
    """Compute the expected output of the 8-bit ALU."""
    op = operation.upper()
    carry = borrow = overflow = 0
    if op == "ADD":
        total = a + b
        result = total & 0xFF
        carry = int(total > 255)
        overflow = int(bool((~(a ^ b) & (a ^ result)) & 0x80))
    elif op == "SUB":
        result = (a - b) & 0xFF
        borrow = int(a < b)
        overflow = int(bool(((a ^ b) & (a ^ result)) & 0x80))
    elif op == "AND":
        result = a & b
    elif op == "OR":
        result = a | b
    else:
        raise ValueError(f"Unknown operation: {operation!r}")
    return {
        "result": result,
        "carry": carry,
        "borrow": borrow,
        "zero": int(result == 0),
        "overflow": overflow,
    }


def check_alu(operation, a, b):
    """Compare the FPGA response with the software reference model."""
    expected = reference_alu(a, b, operation)
    actual = execute_alu(a, b, operation)
    assert actual == expected, (
        f"{operation}({a},{b}): expected={expected}, actual={actual}"
    )
    return actual

## 5. Hardware tests

Start with a single known-good operation, then run the supported operations and flag boundary cases. The controller should return to idle after every test.

In [22]:
print("Smoke test:", check_alu("ADD", 3, 2))

Smoke test: {'result': 5, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}


In [23]:
DIRECTED_TESTS = [
    ("ADD", 3, 2),
    ("SUB", 7, 3),
    ("AND", 0b1100, 0b1010),
    ("OR", 0b1100, 0b1010),
    ("ADD", 255, 1),   # Carry and zero
    ("SUB", 3, 5),     # Borrow
    ("SUB", 7, 7),     # Zero
    ("ADD", 127, 1),   # Positive signed overflow
    ("SUB", 128, 1),   # Signed overflow in subtraction
    ("AND", 0, 255),   # Zero on logical AND
    ("OR", 0, 0),      # Zero on logical OR
]

for op, a, b in DIRECTED_TESTS:
    actual = check_alu(op, a, b)
    print(f"PASS {op}({a}, {b}) -> {actual}", flush=True)

print(f"All {len(DIRECTED_TESTS)} directed tests passed.")

PASS ADD(3, 2) -> {'result': 5, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}
PASS SUB(7, 3) -> {'result': 4, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}
PASS AND(12, 10) -> {'result': 8, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}
PASS OR(12, 10) -> {'result': 14, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 0}
PASS ADD(255, 1) -> {'result': 0, 'carry': 1, 'borrow': 0, 'zero': 1, 'overflow': 0}
PASS SUB(3, 5) -> {'result': 254, 'carry': 0, 'borrow': 1, 'zero': 0, 'overflow': 0}
PASS SUB(7, 7) -> {'result': 0, 'carry': 0, 'borrow': 0, 'zero': 1, 'overflow': 0}
PASS ADD(127, 1) -> {'result': 128, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 1}
PASS SUB(128, 1) -> {'result': 127, 'carry': 0, 'borrow': 0, 'zero': 0, 'overflow': 1}
PASS AND(0, 255) -> {'result': 0, 'carry': 0, 'borrow': 0, 'zero': 1, 'overflow': 0}
PASS OR(0, 0) -> {'result': 0, 'carry': 0, 'borrow': 0, 'zero': 1, 'overflow': 0}
All 11 directed tests passed.


## 6. Optional repeatable random tests

Disabled by default. This is a small *hardware sample*, not a substitute for exhaustive GHDL simulation.

In [ ]:
RUN_RANDOM_TESTS = False
RANDOM_CASES_PER_OPERATION = 8

if RUN_RANDOM_TESTS:
    rng = Random(0xA1A)
    passed = 0
    for operation in OPCODES:
        for _ in range(RANDOM_CASES_PER_OPERATION):
            a, b = rng.randrange(256), rng.randrange(256)
            check_alu(operation, a, b)
            passed += 1
    print(f"All {passed} random hardware tests passed.")
else:
    print("Random tests skipped; set RUN_RANDOM_TESTS=True to enable.")

## Notes

- Platform: PYNQ-Z2 / Zynq-7020; FCLK0 = 50 MHz.
- Programming: Vivado JTAG, not `Overlay.download()`.
- Interface: two AXI GPIO IPs; fixed direction set in Vivado.
- Coverage: directed operations, status flags, and optionally random cases.
- Limitation: Python timeouts cannot interrupt stalled AXI bus transactions.

Recheck the bitstream and AXI address map if the Vivado design changes.